In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Define paths

W10_MAIN_PATH = "path/to/wave_10_elsa_data_eul_v4.dta"

W10_FIN_PATH = "path/to/wave_10_financial_derived_variables.dta"

W11_MAIN_PATH = "path/to/wave_11_elsa_data_eul.dta"

W11_FIN_PATH = "path/to/wave_11_financial_derived_variables.dta"

ROOT = Path.home() / "Documents/PhD/paper_2"
ELSA_DIR = ROOT / "data/elsa"

W10_MAIN_PATH = ELSA_DIR / "wave_10_elsa_data_eul_v4.dta"
# path to ELSA data EUL (Wave 10)

W10_FIN_PATH = ELSA_DIR / "wave_10_financial_derived_variables.dta"
# path to ELSA financial derived variables (Wave 10)

W11_MAIN_PATH = ELSA_DIR / "wave_11_elsa_data_eul.dta"
# path to ELSA data EUL (Wave 11)

W11_FIN_PATH = ELSA_DIR / "wave_11_financial_derived_variables.dta"
# path to ELSA financial derived variables (Wave 11)

In [4]:
# Define variables of interest 

core_cols = [
    "idauniq",
    "corepartner",
    "iapid",
    "gor",
    "indager",
    "indsex",
    "couple",
    "fqethnmr",
    "fqcbthmr",
    "fqaqua",
    "hotenu",

    # ADLs
    "headldr",
    "headlwa",
    "headlba",
    "headlea",
    "headlbe",
    "headlwc",

    # IADLs
    "headlma",
    "headlda",
    "headlpr",
    "headlsh",
    "headlph",
    "headlsp",
    "headlme",
    "headlho",
    "headlmo",
    "headl96"
]

health_cols = [
    "hehelf",
]


fin_cols = [
    "idauniq",
    "nettotnhw_bu_s",
    "tnhwq5_bu_s"
]

w10_specific_cols = [
    "w10xwgt",
    "idahhw10",
    "w10nssec8",
    "w10nssec5",
    "w10nssec3",
    "w10soc2000r",
    "w10sic2003r",
]


w11_specific_cols = [
    "w11xwgt",
    "idahhw11",
    "w11nssec8",
    "w11nssec5",
    "w11nssec3",
    "w11soc2000r",
    "w11sic2003r",
]


common_main_cols = (
    core_cols
    + health_cols
)


w10_main_cols = (
    common_main_cols
    + w10_specific_cols
)

w11_main_cols = (
    common_main_cols
    + w11_specific_cols
)

In [5]:
# Load ELSA files

w10_main = pd.read_stata(
    W10_MAIN_PATH,
    columns=w10_main_cols,
    convert_categoricals=False,
)

w10_fin = pd.read_stata(
    W10_FIN_PATH,
    columns=fin_cols,
    convert_categoricals=False,
)


w11_main = pd.read_stata(
    W11_MAIN_PATH,
    columns=w11_main_cols,
    convert_categoricals=False,
)

w11_fin = pd.read_stata(
    W11_FIN_PATH,
    columns=fin_cols,
    convert_categoricals=False,
)

In [6]:
# Merge wave 10 with wave 11


w10 = w10_main.merge(
    w10_fin,
    on="idauniq",
    how="left",
    validate="one_to_one",
)


w11 = w11_main.merge(
    w11_fin,
    on="idauniq",
    how="left",
    validate="one_to_one",
)

In [7]:
# Make naming conventions standard

w10 = w10.rename(
    columns={
        "w10xwgt": "xwgt",
        "idahhw10": "hhid",
        "w10nssec8": "nssec8",
        "w10nssec5": "nssec5",
        "w10nssec3": "nssec3",
        "w10soc2000r": "soc2000r",
        "w10sic2003r": "sic2003r",
    }
)


w11 = w11.rename(
    columns={
        "w11xwgt": "xwgt",
        "idahhw11": "hhid",
        "w11nssec8": "nssec8",
        "w11nssec5": "nssec5",
        "w11nssec3": "nssec3",
        "w11soc2000r": "soc2000r",
        "w11sic2003r": "sic2003r",
    }
)


w10["wave"] = 10
w11["wave"] = 11

In [8]:
# Handle missing codes

ELSA_MISSING_CODES = [-9, -8, -2, -1]

PROTECTED = ["idauniq", "wave", "nettotnhw_bu_s"]

cols = w10.select_dtypes("number").columns.difference(PROTECTED)
w10[cols] = w10[cols].mask(w10[cols].isin(ELSA_MISSING_CODES))

cols = w11.select_dtypes("number").columns.difference(PROTECTED)

w11[cols] = w11[cols].mask(w11[cols].isin(ELSA_MISSING_CODES))

In [9]:
# Check number of respondents in both waves

print(
    f"\nWave 10 respondents: "
    f"{w10['idauniq'].nunique():,}"
)

print(
    f"Wave 11 respondents: "
    f"{w11['idauniq'].nunique():,}"
)

ids10 = set(w10["idauniq"])
ids11 = set(w11["idauniq"])

both_ids = ids10 & ids11

only10_ids = ids10 - ids11
only11_ids = ids11 - ids10


print(
    f"\nObserved in both waves: "
    f"{len(both_ids):,}"
)

print(
    f"Wave 10 only: "
    f"{len(only10_ids):,}"
)

print(
    f"Wave 11 only: "
    f"{len(only11_ids):,}"
)

only_w10 = sorted(
    set(w10.columns)
    - set(w11.columns)
)

only_w11 = sorted(
    set(w11.columns)
    - set(w10.columns)
)

w11 = w11[w10.columns]



Wave 10 respondents: 7,589
Wave 11 respondents: 7,842

Observed in both waves: 5,584
Wave 10 only: 2,005
Wave 11 only: 2,258


In [10]:
# Sort values

elsa_long = pd.concat(
    [
        w10,
        w11,
    ],
    ignore_index=True,
    sort=False,
)


elsa_long = (
    elsa_long
    .sort_values(
        [
            "idauniq",
            "wave",
        ]
    )
    .reset_index(drop=True)
)


In [11]:
# Drop duplicates


value_cols = [
    col
    for col in elsa_long.columns
    if col not in [
        "idauniq",
        "wave",
    ]
]


elsa_filled = elsa_long.copy()


elsa_filled[value_cols] = (
    elsa_filled
    .groupby(
        "idauniq",
        sort=False,
    )[value_cols]
    .ffill()
)

elsa = (
    elsa_filled
    .drop_duplicates(
        subset="idauniq",
        keep="last",
    )
    .copy()
    .reset_index(drop=True)
)


elsa = elsa.rename(
    columns={
        "wave": "latest_wave",
    }
)

In [12]:
# Filter

elsa65 = elsa.query("corepartner == 1").loc[
    (elsa["indager"] >= 65)
    | (elsa["indager"] == -7)
].copy()


Sex

In [13]:
# Define sex variables

elsa65["male"] = np.where(
    elsa65["indsex"].isin([-9, -8, -7, -6, -5, -4, -3, -2, -1]),
    np.nan,
    np.where(elsa65["indsex"] == 1, 1, 0).astype(int)
)

elsa65["female"] = np.where(
    elsa65["indsex"].isin([-9, -8, -7, -6, -5, -4, -3, -2, -1]),
    np.nan,
    np.where(elsa65["indsex"] == 2, 1, 0).astype(int)
)

elsa65[["idauniq", "male", "female", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/sex.csv")

elsa65[["idauniq", "male", "female", "xwgt", "hhid", "gor"]].head()

,idauniq,male,female,xwgt,hhid,gor
29,103712,0.0,1.0,0.544087,107605,E12000003
30,103724,0.0,1.0,0.407471,102781,E12000008
31,103725,1.0,0.0,0.493833,109562,E12000008
32,103726,1.0,0.0,0.685498,106642,E12000005
33,103727,0.0,1.0,0.437373,114114,E12000001


Age

In [14]:
elsa65["indager"] = pd.to_numeric(
    elsa65["indager"],
    errors="coerce"
)

elsa65["age_65_69"] = (
    elsa65["indager"].between(65, 69)
).astype(int)

elsa65["age_70_74"] = (
    elsa65["indager"].between(70, 74)
).astype(int)

elsa65["age_75_79"] = (
    elsa65["indager"].between(75, 79)
).astype(int)

elsa65["age_80_84"] = (
    elsa65["indager"].between(80, 84)
).astype(int)

# 85+ combines ages 85–89 and 90+

elsa65["age_85_plus"] = (
    elsa65["indager"].between(85, 89)
    | (elsa65["indager"] == -7)
).astype(int)



elsa65[[
    "idauniq", "xwgt", "hhid", "gor",
    "age_65_69",
    "age_70_74",
    "age_75_79",
    "age_80_84",
    "age_85_plus"]].to_csv("../data/elsa/age.csv")

Ethnicity

In [15]:
elsa65["ethnicity_white"] = np.where(
    elsa65["fqethnmr"].isna(),
    np.nan,
    (elsa65["fqethnmr"] == 1).astype(int)
)

elsa65["ethnicity_non_white"] = np.where(
    elsa65["fqethnmr"].isna(),
    np.nan,
    (elsa65["fqethnmr"] == 2).astype(int)
)

elsa65[["idauniq", "ethnicity_white", "ethnicity_non_white", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/ethnicity.csv")

elsa65.head()

,idauniq,corepartner,iapid,gor,indager,indsex,couple,fqethnmr,fqcbthmr,fqaqua,...,latest_wave,male,female,age_65_69,age_70_74,age_75_79,age_80_84,age_85_plus,ethnicity_white,ethnicity_non_white
29,103712,1,2.0,E12000003,75,2,3,1.0,1.0,NaN,...,10,0.0,1.0,0,0,1,0,0,1.0,0.0
30,103724,1,1.0,E12000008,87,2,3,1.0,1.0,NaN,...,10,0.0,1.0,0,0,0,0,1,1.0,0.0
31,103725,1,1.0,E12000008,73,1,1,1.0,1.0,NaN,...,10,1.0,0.0,0,1,0,0,0,1.0,0.0
32,103726,1,1.0,E12000005,75,1,3,1.0,1.0,NaN,...,11,1.0,0.0,0,0,1,0,0,1.0,0.0
33,103727,1,2.0,E12000001,75,2,1,1.0,1.0,1.0,...,11,0.0,1.0,0,0,1,0,0,1.0,0.0


Health

In [16]:
elsa65["hehelf"] = pd.to_numeric(
    elsa65["hehelf"],
    errors="coerce"
)

HEALTH_COLS = ["health_very_good_health", "health_good_health",
               "health_fair_health", "health_bad_health"]

elsa65["health_very_good_health"] = (
    elsa65["hehelf"].isin([1, 2])
).astype(int)

elsa65["health_good_health"] = (
    elsa65["hehelf"] == 3
).astype(int)

elsa65["health_fair_health"] = (
    elsa65["hehelf"] == 4
).astype(int)

# Poor (5) matches the Census "bad health" and "very bad health"

elsa65["health_bad_health"] = (
    elsa65["hehelf"] == 5
).astype(int)

health_summary = pd.DataFrame({
    "count": elsa65[HEALTH_COLS].sum(),
    "proportion": elsa65[HEALTH_COLS].mean()
})


elsa65[[
    "idauniq", "xwgt", "hhid", "gor",
    *HEALTH_COLS]].to_csv("../data/elsa/health.csv", index=False)

health_summary.head()


,count,proportion
health_very_good_health,1502,0.3004
health_good_health,1709,0.3418
health_fair_health,1121,0.2242
health_bad_health,498,0.0996


NSSEC

In [18]:
elsa65["nssec_managerial"] = (elsa65["nssec3"] == 1).astype(int)
elsa65["nssec_intermediate"] = (elsa65["nssec3"] == 2).astype(int)
elsa65["nssec_manual"] = (elsa65["nssec3"] == 3).astype(int)

elsa65[["idauniq", "nssec_managerial", "nssec_intermediate", "nssec_manual", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/nssec.csv")

elsa65[["idauniq", "nssec_managerial", "nssec_intermediate", "nssec_manual", "xwgt", "hhid", "gor"]].head()

,idauniq,nssec_managerial,nssec_intermediate,nssec_manual,xwgt,hhid,gor
29,103712,0,0,0,0.544087,107605,E12000003
30,103724,0,0,0,0.407471,102781,E12000008
31,103725,0,0,0,0.493833,109562,E12000008
32,103726,0,0,1,0.685498,106642,E12000005
33,103727,0,0,1,0.437373,114114,E12000001


Country of birth

In [19]:
elsa65["uk_born"] = np.where(
    elsa65["fqcbthmr"].isna(),
    np.nan,
    (elsa65["fqcbthmr"] == 1).astype(int)
)

elsa65["non_uk_born"] = np.where(
    elsa65["fqcbthmr"].isna(),
    np.nan,
    (elsa65["fqcbthmr"] == 2).astype(int)
)
elsa65[["idauniq", "xwgt", "hhid", "gor", "non_uk_born", "uk_born"]].to_csv("../data/elsa/country_of_birth.csv")

elsa65[["idauniq", "xwgt", "hhid", "gor", "non_uk_born", "uk_born"]].head()

,idauniq,xwgt,hhid,gor,non_uk_born,uk_born
29,103712,0.544087,107605,E12000003,0.0,1.0
30,103724,0.407471,102781,E12000008,0.0,1.0
31,103725,0.493833,109562,E12000008,0.0,1.0
32,103726,0.685498,106642,E12000005,0.0,1.0
33,103727,0.437373,114114,E12000001,0.0,1.0


Partnership

In [20]:
elsa65["partner_has_partner"] = np.where(
    elsa65["couple"].isin([1, 2]), 1,
    np.where(elsa65["couple"] == 3, 0, np.nan).astype(int)
)
elsa65["partner_does_not_have_partner"] = np.where(
    elsa65["partner_has_partner"] == 0, 1, 0).astype(int)


elsa65[["idauniq", "partner_has_partner", "partner_does_not_have_partner",  "xwgt", "hhid", "gor"]].to_csv("../data/elsa/partnership.csv")

elsa65[["idauniq", "partner_has_partner", "partner_does_not_have_partner",  "xwgt", "hhid", "gor"]].head()

/tmp/ipykernel_5465/1443947970.py:3: RuntimeWarning: invalid value encountered in cast
  np.where(elsa65["couple"] == 3, 0, np.nan).astype(int)


,idauniq,partner_has_partner,partner_does_not_have_partner,xwgt,hhid,gor
29,103712,0,1,0.544087,107605,E12000003
30,103724,0,1,0.407471,102781,E12000008
31,103725,1,0,0.493833,109562,E12000008
32,103726,0,1,0.685498,106642,E12000005
33,103727,1,0,0.437373,114114,E12000001


Home ownership

In [21]:
elsa65["homeowner"] = np.where(
    elsa65["hotenu"].isin([1, 2, 3]), 1,
    np.where(elsa65["hotenu"].isin([4, 5, 6]), 0, np.nan)
)

elsa65[["idauniq", "homeowner", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/homeownership.csv")

Non-housing wealth quintiles

In [22]:
elsa65["tnhwq5_bu_s"]

for q in [1, 2, 3, 4, 5]:
    elsa65[f"wealth_q{q}"] = np.where(
        elsa65["tnhwq5_bu_s"].isna(),
        np.nan,
        np.where(elsa65["tnhwq5_bu_s"] == q, 1, 0)
    )

elsa65[["idauniq", "wealth_q1","wealth_q2", "wealth_q3", "wealth_q4","wealth_q5", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/wealth.csv")

Care need

In [23]:
ADL_VARS = [
    "headldr",  # dressing, including putting on shoes and socks
    "headlwa",  # walking across a room
    "headlba",  # bathing or showering
    "headlea",  # eating, such as cutting up food
    "headlbe",  # getting in or out of bed
    "headlwc",  # using the toilet, including getting up or down
]

IADL_VARS = [
    "headlma",  # using a map to figure out how to get around
    "headlpr",  # preparing a hot meal
    "headlsh",  # shopping for groceries
    "headlph",  # using the telephone
    "headlme",  # taking medications
    "headlho",  # doing work around the house or garden
    "headlmo",  # managing money, e.g. paying bills
    "headlda",  # recognising when in physical danger
    "headlsp",  # communication (speech, hearing or eyesight)
]

ADL_IADL_VARS = ADL_VARS + IADL_VARS

elsa65["n_adl"]  = elsa65[ADL_VARS].sum(axis=1)
elsa65["n_iadl"] = elsa65[IADL_VARS].sum(axis=1)
elsa65["n_limitations"] = elsa65["n_adl"] + elsa65["n_iadl"]


elsa65["has_any_care_needs"]  = (elsa65["n_limitations"] > 0).astype(int)
elsa65["has_multiple_needs"]  = (elsa65["n_limitations"] > 1).astype(int)
elsa65["has_no_needs"]        = (elsa65["headl96"] == 1).astype(int)

elsa65[["idauniq", "has_any_care_needs", "n_limitations", "has_multiple_needs", "has_no_needs", "xwgt", "hhid", "gor"]].to_csv("../data/elsa/care_need.csv")